## Linear Projection 

- In this section we are applying an affine transformation where we convert (prepare) our tokens ``C`` dimensional vector into `3C`
- This will then be split into 3 components used in the attention mechanism
- We start from ``(B, T, C)`` matmul with ``(C, 3C)`` producing ``(B, T, 3C)``

In [11]:
import torch 
import torch.nn as nn

In [12]:
class QKVProjection(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        
        self.qkv = nn.Linear(n_embd, 3*n_embd)

    def forward(self, x):
        return self.qkv(x)

In [13]:
proj = QKVProjection(4)

out = proj(torch.tensor([[[1.0, 2.0, 3.0, 4.0]]])) # Given the dimension is (B, T, C) (1, 1, 4) what could this represent? 

print(out.shape)

torch.Size([1, 1, 12])


# Bringing it together till now

In [16]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.ln = nn.LayerNorm(n_embd)
        self.qkv = nn.Linear(n_embd, 3*n_embd)



    def forward(self, idx):
        B, T = idx.shape
        token_emb = self.wte(idx)
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        x = token_emb + pos_emb 
        norm_x = self.ln(x)
        proj_x = self.qkv(norm_x)
        return proj_x


In [17]:
model = Transformer(vocab_size=4, n_embd=6, block_size=4)
idx = torch.tensor([[0, 1, 2, 3]])
print(idx.shape)
out = model(idx)
print(out)
print(out.shape)

torch.Size([1, 4])
tensor([[[ 0.3666, -0.8132,  0.0464, -0.1163, -0.3476, -0.2652,  0.8046,
          -0.1154, -1.3142,  0.7937,  0.8653, -0.5638,  0.6443, -0.0770,
          -0.1620,  0.4009,  0.3802,  0.3307],
         [ 0.2332,  0.1920,  0.3243,  0.0056, -0.9341, -0.9806,  0.2247,
           0.4690,  0.1625,  0.6085,  0.4970, -0.2416, -0.6722, -0.0143,
           0.0528, -0.7164, -0.5664, -0.4473],
         [ 0.1442,  0.6586, -0.1441, -0.1061, -0.0860,  0.0579, -0.8949,
          -0.1004,  0.2355,  0.0791, -0.6329,  0.2917,  0.7556, -0.9393,
           0.8213, -0.5552,  0.0752, -1.1685],
         [-0.0280,  0.8076, -0.5100, -0.7117, -0.6387, -0.9392, -0.4186,
           0.3498,  0.4236, -0.2403,  0.3903,  0.8032, -0.0449, -0.1030,
           0.1549, -0.7890,  0.0356, -0.9165]]], grad_fn=<ViewBackward0>)
torch.Size([1, 4, 18])
